# Donut Inference Speed Ablation

Interprets `results/ablation/bench_speed.json` produced by `scripts/run_ablation.sh`.

Questions answered:
1. **Which backend is fastest?** — absolute throughput + 95% CI
2. **Does the speedup grow with resolution?** — tells us whether bigger images benefit more from acceleration
3. **What batch size maximises throughput?** — throughput vs memory tradeoff
4. **Where does time go?** — encoder vs decoder split
5. **Decision table** — ranked configs with confidence intervals

In [ ]:
import json
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

mpl.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

RESULTS = Path("../results/ablation")
raw = json.loads((RESULTS / "bench_speed.json").read_text())
print(json.dumps(raw["meta"], indent=2))

df = pd.json_normalize(raw["records"])
df["image_label"]  = df["image_height"].astype(str) + "×" + df["image_width"].astype(str)
df["image_pixels"] = df["image_height"] * df["image_width"]
# 95% confidence-interval half-width: 1.96 * σ / √n
df["ci95_gen"] = 1.96 * df["generate.std_ms"]  / np.sqrt(df["generate.n_runs"])
df["ci95_enc"] = 1.96 * df["encoder.std_ms"]   / np.sqrt(df["encoder.n_runs"])

# Ordered display lists (drop baseline from backend list for most charts)
ALL_BACKENDS = [b for b in ["baseline", "eager", "sdpa", "fa"] if b in df["backend"].unique()]
BACKENDS     = [b for b in ALL_BACKENDS if b != "baseline"]
SIZES        = sorted(df["image_pixels"].unique())
SIZE_LABELS  = [df[df["image_pixels"] == px]["image_label"].iloc[0] for px in SIZES]

COLORS = {"baseline": "#aaaaaa", "eager": "#4C72B0", "sdpa": "#DD8452", "fa": "#55A868"}

print(f"\nBackends in data : {ALL_BACKENDS}")
print(f"Image sizes      : {SIZE_LABELS}")
print(f"Batch sizes      : {sorted(df['batch_size'].unique())}")
df.head()

## 1. Which backend is fastest?

Generate throughput (images/s) at **batch_size=1**, grouped by image resolution.  
Error bars = **95% confidence interval** (`1.96 σ/√n`).  
A difference is reliable when the error bars of two bars do not overlap.

In [ ]:
bs1 = df[df["batch_size"] == 1]

fig, axes = plt.subplots(1, len(SIZES), figsize=(5 * len(SIZES), 4), sharey=False)
if len(SIZES) == 1:
    axes = [axes]

for ax, (px, lbl) in zip(axes, zip(SIZES, SIZE_LABELS)):
    sub = bs1[bs1["image_pixels"] == px]
    for i, be in enumerate(ALL_BACKENDS):
        row = sub[sub["backend"] == be]
        if row.empty:
            continue
        val  = row["throughput.images_per_s"].values[0]
        ci   = row["ci95_gen"].values[0]
        bar  = ax.bar(i, val, color=COLORS.get(be, "#999"), width=0.6, zorder=3)
        ax.errorbar(i, val, yerr=ci, fmt="none", color="black", capsize=4, linewidth=1.5, zorder=4)
        ax.text(i, val + ci + 0.2, f"{val:.1f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(range(len(ALL_BACKENDS)))
    ax.set_xticklabels(ALL_BACKENDS, rotation=15, ha="right")
    ax.set_ylabel("images / s" if ax is axes[0] else "")
    ax.set_title(lbl)
    ax.yaxis.grid(True, linestyle="--", alpha=0.4, zorder=0)

plt.suptitle("Generate throughput by backend  (batch_size=1, ±95% CI)", y=1.02)
plt.tight_layout()
plt.show()

# Takeaway
target_px = max(SIZES)  # use the largest image size as the reference
target_lbl = SIZE_LABELS[SIZES.index(target_px)]
sub_target = bs1[bs1["image_pixels"] == target_px]
best_row   = sub_target[sub_target["backend"].isin(BACKENDS)].sort_values(
    "throughput.images_per_s", ascending=False
).iloc[0]
eager_row  = sub_target[sub_target["backend"] == "eager"]
eager_val  = eager_row["throughput.images_per_s"].values[0] if not eager_row.empty else float("nan")
speedup    = best_row["throughput.images_per_s"] / eager_val
print(f"\n→ Fastest backend at {target_lbl}: {best_row['backend']}  "
      f"{best_row['throughput.images_per_s']:.1f} img/s  "
      f"({speedup:.1f}× vs eager)")

## 2. Does the speedup grow with resolution?

Speedup of each backend vs the no-acceleration baseline, plotted across image sizes.  
If lines slope upward → the backend benefits **more** from acceleration at larger images  
(more attention windows = more to accelerate). Flat lines → resolution-agnostic gain.

In [ ]:
bs1 = df[df["batch_size"] == 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (col, title) in zip(axes, [
    ("speedup_vs_baseline.encoder", "Encoder speedup vs baseline"),
    ("speedup_vs_baseline.generate", "Generate speedup vs baseline"),
]):
    for be in ALL_BACKENDS:
        sub = bs1[bs1["backend"] == be].sort_values("image_pixels")
        if sub.empty:
            continue
        ax.plot(
            sub["image_pixels"], sub[col],
            "o-", label=be, color=COLORS.get(be, "#999"), linewidth=2, markersize=7,
        )
    ax.axhline(1.0, color="gray", linewidth=0.8, linestyle="--")
    ax.set_xticks(SIZES)
    ax.set_xticklabels(SIZE_LABELS, rotation=10)
    ax.set_xlabel("image size")
    ax.set_ylabel("speedup vs baseline")
    ax.set_title(title)
    ax.yaxis.grid(True, linestyle="--", alpha=0.4)
    ax.legend(fontsize=9)

plt.suptitle("Backend speedup across image resolutions  (batch_size=1)", y=1.02)
plt.tight_layout()
plt.show()

# Takeaway: does speedup grow with resolution for the best backend?
if len(SIZES) >= 2:
    for be in BACKENDS:
        sub = bs1[bs1["backend"] == be].sort_values("image_pixels")
        if sub.empty or len(sub) < 2:
            continue
        lo = sub.iloc[0]["speedup_vs_baseline.generate"]
        hi = sub.iloc[-1]["speedup_vs_baseline.generate"]
        lo_lbl = SIZE_LABELS[0]; hi_lbl = SIZE_LABELS[-1]
        trend  = "grows" if hi > lo + 0.05 else ("shrinks" if hi < lo - 0.05 else "is stable")
        print(f"→ {be}: generate speedup {trend} with resolution  "
              f"({lo_lbl}: {lo:.2f}× → {hi_lbl}: {hi:.2f}×)")

## 3. What batch size maximises throughput?

Higher batch = better GPU utilisation, but more memory.  
The curve flattening tells you the **optimal batch size** given your GPU memory budget.

In [ ]:
fig, axes = plt.subplots(1, len(SIZES), figsize=(5 * len(SIZES), 4), sharey=False)
if len(SIZES) == 1:
    axes = [axes]

for ax, (px, lbl) in zip(axes, zip(SIZES, SIZE_LABELS)):
    sub = df[df["image_pixels"] == px]
    for be in BACKENDS:
        grp = sub[sub["backend"] == be].sort_values("batch_size")
        if grp.empty:
            continue
        ax.plot(
            grp["batch_size"], grp["throughput.images_per_s"],
            "o-", label=be, color=COLORS.get(be, "#999"), linewidth=2, markersize=7,
        )
    ax.set_xlabel("batch size")
    ax.set_ylabel("images / s" if ax is axes[0] else "")
    ax.set_title(lbl)
    ax.set_xticks(sorted(df["batch_size"].unique()))
    ax.yaxis.grid(True, linestyle="--", alpha=0.4)
    ax.legend(fontsize=9)

plt.suptitle("Throughput vs batch size per image resolution", y=1.02)
plt.tight_layout()
plt.show()

# Takeaway: at the largest image size, where does the best backend peak?
target_px  = max(SIZES)
target_lbl = SIZE_LABELS[SIZES.index(target_px)]
best_be    = best_row["backend"]  # from cell 1
grp = df[(df["image_pixels"] == target_px) & (df["backend"] == best_be)].sort_values("batch_size")
if not grp.empty:
    peak_bs  = grp.loc[grp["throughput.images_per_s"].idxmax(), "batch_size"]
    peak_val = grp["throughput.images_per_s"].max()
    print(f"\n→ Best throughput for {best_be} at {target_lbl}: "
          f"{peak_val:.1f} img/s at batch_size={peak_bs}")

## 4. Where does time go? Encoder vs decoder split

Stacked bars: bottom = encoder, top = decoder (generate − encoder).  
This shows whether further wins should come from encoder or decoder optimisation.

In [ ]:
target_px  = max(SIZES)
target_lbl = SIZE_LABELS[SIZES.index(target_px)]
bs1_target = df[(df["batch_size"] == 1) & (df["image_pixels"] == target_px)]

fig, ax = plt.subplots(figsize=(7, 4))

x = range(len(ALL_BACKENDS))
enc_vals = []
dec_vals = []
for be in ALL_BACKENDS:
    row = bs1_target[bs1_target["backend"] == be]
    if row.empty:
        enc_vals.append(0); dec_vals.append(0)
        continue
    enc_ms = row["encoder.mean_ms"].values[0]
    gen_ms = row["generate.mean_ms"].values[0]
    enc_vals.append(enc_ms)
    dec_vals.append(max(gen_ms - enc_ms, 0))

ax.bar(x, enc_vals, label="encoder", color="#4C72B0", zorder=3)
ax.bar(x, dec_vals, bottom=enc_vals, label="decoder (generate − encoder)",
       color="#DD8452", zorder=3)

for i, (e, d) in enumerate(zip(enc_vals, dec_vals)):
    total = e + d
    if total > 0:
        enc_pct = 100 * e / total
        ax.text(i, total + 1, f"{total:.0f}ms", ha="center", fontsize=8)
        ax.text(i, e / 2, f"{enc_pct:.0f}%", ha="center", va="center",
                color="white", fontsize=8, fontweight="bold")

ax.set_xticks(list(x))
ax.set_xticklabels(ALL_BACKENDS)
ax.set_ylabel("ms  (batch_size=1)")
ax.set_title(f"Encoder vs decoder latency — {target_lbl}")
ax.legend(fontsize=9)
ax.yaxis.grid(True, linestyle="--", alpha=0.4, zorder=0)
plt.tight_layout()
plt.show()

# Takeaway
best_be_row = bs1_target[bs1_target["backend"] == best_row["backend"]]
if not best_be_row.empty:
    enc_ms = best_be_row["encoder.mean_ms"].values[0]
    gen_ms = best_be_row["generate.mean_ms"].values[0]
    enc_pct = 100 * enc_ms / gen_ms
    print(f"\n→ For {best_row['backend']} at {target_lbl}: "
          f"encoder = {enc_pct:.0f}% of total generate latency  "
          f"({enc_ms:.0f}ms / {gen_ms:.0f}ms)")

## 5. Decision table

All (backend, image_size, batch_size) configurations ranked by throughput.  
**The top row is the recommended config.** The ±CI column shows statistical reliability —  
configs whose ranges overlap are not reliably different from each other.

In [ ]:
out = df[df["backend"] != "baseline"].copy()
out["img/s"]      = out["throughput.images_per_s"].round(2)
out["± 95% CI"]   = out["ci95_gen"].round(2)
out["enc ms"]     = out["encoder.mean_ms"].round(1)
out["gen ms"]     = out["generate.mean_ms"].round(1)
out["gen speedup"] = out["speedup_vs_baseline.generate"].round(2)

table = out.sort_values("throughput.images_per_s", ascending=False)[
    ["image_label", "backend", "batch_size",
     "img/s", "± 95% CI", "gen speedup", "enc ms", "gen ms"]
].reset_index(drop=True)

print(table.to_string(index=True))

print("\n" + "="*60)
top = table.iloc[0]
print(f"RECOMMENDATION:  backend={top['backend']}  "
      f"image={top['image_label']}  bs={top['batch_size']}")
print(f"  → {top['img/s']} ± {top['± 95% CI']} img/s  "
      f"({top['gen speedup']}× vs baseline)")
print("="*60)